# B.Y.O.D - Use `pygsdata` for your own data!

In this session, the idea is to try and use `pygsdata` to read your own data and make some initial plots. Here are some things to think about as you try this:

* If this process is relatively simple, would it be worth trying to use `pygsdata` in your experiment?
* What are the main pain-points of this exercise? Are there aspects of your data that make the structure of a `GSData` object either impossible to use, or more difficult than it has to be? Potential examples might be:
    * You have multiple loads in a file, but each load is at a different cadence
    * You point your telescope over time
    * You have frequency-dependent metadata that you'd like to track
* Are there any bugs you encounter?
* What does `pygsdata` do right? Does it "feel" natural to use? 

The rest of this notebook will simply be a set of empty sections with prompts to give you something to try to achieve:

In [7]:
from pygsdata import GSData, gsdata_reader, gsregister
from pygsdata import plots
from pathlib import Path

## 1. Read your data as a pygsdata object

The first thing to do is just create a `GSData` object that contains your data (from whatever file format it is in). You might want to keep the `data_loading_writing` notebook open, as it outlines the relevant quantities you need to specify. However, you can also just read the list:

In [2]:
GSData?

Init signature:
GSData(
    telescope: pygsdata.telescope.Telescope,
    data: ArrayLike | astropy.units.quantity.Quantity | None,
    freqs: ArrayLike | astropy.units.quantity.Quantity | None,
    times: astropy.time.core.Time,
    pols: collections.abc.Sequence[str] = NOTHING,
    effective_integration_time: ArrayLike | astropy.units.quantity.Quantity | None = NOTHING,
    nsamples: ArrayLike | astropy.units.quantity.Quantity | None = NOTHING,
    loads=NOTHING,
    flags: dict[str, pygsdata.gsflag.GSFlag] = NOTHING,
    history: pygsdata.history.History = NOTHING,
    residuals: ArrayLike | astropy.units.quantity.Quantity | None = None,
    data_unit: Literal['power', 'temperature', 'uncalibrated', 'uncalibrated_temp'] = 'power',
    auxiliary_measurements=None,
    time_ranges: astropy.time.core.Time = NOTHING,
    lsts=NOTHING,
    lst_ranges=NOTHING,
    filename=None,
    file_appendable=True,
    name='',
) -> None
Docstring:     
A generic container for Global-Signal data.

Pa

## 2. Turn your reader into a proper "reader"

Re-package the code you have above into a simple function and decorate it with the `gsdata_reader` decorator:

In [ ]:
@gsdata_reader(
    select_on_read=False,  # you can set this to True if you're able to do partial I/O on your files!
    formats=['YOUR_FORMAT_HERE']
)
def read_my_format(path: Path) -> GSData:
    ...
    return GSData(
        # Insert the attributes you've read here!
    )

In [ ]:
raw_data = read_my_format(DATA_PATH)

## 3. Do a simple calibration

You calibrate your own data in your own special way. Perhaps you do Dicke-switching like EDGES, or perhaps you use a noise diode. The aim in this section is to write a new calibration function that does "simple" calibration (for EDGES, this would be our first step of taking the quotient of Dicke switch positions, and multiplying by a rough assumed internal load+noise source temperature). 

In [5]:
@gsregister
def do_simple_calibration(data: GSData) -> GSData:
    # Do your calibration here...
    return data.update(
        data = # your calibrated data...
        # you may also need to update the loads (if you've combined loads),
        # and times arrays. 
    )

SyntaxError: expected argument value expression (1058969775.py, line 5)

In [ ]:
calibrated_data = do_simple_calibration(raw_data)

## 4. Do some flagging

Now, let's apply some flagging. The aim here is to use pre-existing functions that we've already used before, and apply them to your data. Hopefully, since your data is now in pygsdata format, everything should "just work". But it's good to test these algorithms on new kinds of data!

In [6]:
from edges.filters import sun_filter
from edges.filters import rfi_iterative_filter
from edges.filters.xrfi import LinearModeler
from edges import modeling as mdl

In [ ]:
flagged_data = sun_filter(calibrated_data, elevation_range=(-90, 0))

For the RFI filter, you might have to play around with the settings a bit to get it to work. In particular, the linear model that's fit to the data might not be a good fit to your data -- perhaps you may need to increase/decrease number of terms, for example. Perhaps you want to try out different linear models (or write your own!). 

In [ ]:
flagged_data = rfi_iterative_filter(
    flagged_data,
    data_modeler = LinearModeler(            
        model = mdl.Fourier(n_terms=37, period=1.5, transform=mdl.ZerotooneTransform(
            range=(
                d.freqs.min().to_value("MHz"), 
                d.freqs.max().to_value("MHz"),
            )
        ))
    ),
    std_modeler=LinearModeler(            
        model = mdl.Fourier(n_terms=5, period=1.5, transform=mdl.ZerotooneTransform(
            range=(
                d.freqs.min().to_value("MHz"), 
                d.freqs.max().to_value("MHz"),
            )
        ))
    ),
    max_iter=100,
    threshold_setter = lambda x: 2.5,
    watershed = {
        1.0: 4,
        10.0: 8,
        100.0: 16
    },
)

## 5. Make a waterfall plot

If everything has gone well, you should be able to now make a waterfall plot:

In [ ]:
plots.plot_waterfall(flagged_data)

## 6. LST Bin

Let's average the data into 15-minute bins of LST:

In [8]:
from edges.averaging import lst_bin

Note that we haven't yet added a model to the data, so the following will not be corrected for frequency-dependent flagging effects. You may want to do this yourself!

In [ ]:
lstbin_data = lst_bin(
    flagged_data,
    binsize = 0.25,  # hours
    first_edge = 0, # hours
    max_edge = 24,  # hours
)

In [ ]:
plots.plot_waterfall(lstbin_data)

## 7. Add a Model and plot the residuals

You might want to play around with what model you think best represents your data. As a start you can try a LinLog model:

In [ ]:
linlog = mdl.LinLog(n_terms=7)

In [ ]:
lstbin_data = add_model(lstbin_data, model=linlog)

In [ ]:
plots.plot_model_residuals_vs_lst(lstbin_data, offset=??)  # choose an offset that makes it look good